In [1]:
import pandas as pd

# Full path to your file (OneDrive desktop)
path = r"C:\Users\Asus\OneDrive\Desktop\project_self_1\MUP_DPR_RY26_P04_V10_DY24_NPI.csv"

# Read ONLY first 1000 rows so it's instant
peek = pd.read_csv(path, nrows=1000)

print("Number of columns:", peek.shape[1])
print()
print(peek.columns.tolist())

Number of columns: 84

['PRSCRBR_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_MI', 'Prscrbr_Crdntls', 'Prscrbr_Ent_Cd', 'Prscrbr_St1', 'Prscrbr_St2', 'Prscrbr_City', 'Prscrbr_State_Abrvtn', 'Prscrbr_State_FIPS', 'Prscrbr_zip5', 'Prscrbr_RUCA', 'Prscrbr_RUCA_Desc', 'Prscrbr_Cntry', 'Prscrbr_Type', 'Prscrbr_Type_src', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Drug_Cst', 'Tot_Day_Suply', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes', 'Brnd_Sprsn_Flag', 'Brnd_Tot_Clms', 'Brnd_Tot_Drug_Cst', 'Gnrc_Sprsn_Flag', 'Gnrc_Tot_Clms', 'Gnrc_Tot_Drug_Cst', 'Othr_Sprsn_Flag', 'Othr_Tot_Clms', 'Othr_Tot_Drug_Cst', 'MAPD_Sprsn_Flag', 'MAPD_Tot_Clms', 'MAPD_Tot_Drug_Cst', 'PDP_Sprsn_Flag', 'PDP_Tot_Clms', 'PDP_Tot_Drug_Cst', 'LIS_Sprsn_Flag', 'LIS_Tot_Clms', 'LIS_Drug_Cst', 'NonLIS_Sprsn_Flag', 'NonLIS_Tot_Clms', 'NonLIS_Drug_Cst', 'Opioid_Tot_Clms', 'Opioid_Tot_Drug_Cst', 'Opioid_To

In [2]:
# Load a focused subset of useful columns (faster than all 84)
cols = ['PRSCRBR_NPI','Prscrbr_Type','Prscrbr_State_Abrvtn','Prscrbr_RUCA_Desc',
        'Tot_Benes','Bene_Avg_Age','Bene_Avg_Risk_Scre','Tot_Clms','Tot_Drug_Cst']
df_prov = pd.read_csv(path, usecols=cols)

print("Total prescribers:", len(df_prov))
print()
print("Top 10 specialties by count:")
print(df_prov['Prscrbr_Type'].value_counts().head(10))

Total prescribers: 1416883

Top 10 specialties by count:
Prscrbr_Type
Nurse Practitioner                                                278577
Physician Assistant                                               137246
Dentist                                                           132825
Internal Medicine                                                 130220
Family Practice                                                   117797
Student in an Organized Health Care Education/Training Program     67996
Emergency Medicine                                                 56372
Optometry                                                          34179
Obstetrics & Gynecology                                            34042
Pharmacist                                                         31027
Name: count, dtype: int64


In [3]:
# Peek at the second file's columns first (instant - 5 rows only)
path2 = r"C:\Users\Asus\OneDrive\Desktop\project_self_1\MUP_DPR_RY26_P04_V10_DY24_NPIBN.csv"
peek2 = pd.read_csv(path2, nrows=5)
print(peek2.columns.tolist())

['Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City', 'Prscrbr_State_Abrvtn', 'Prscrbr_State_FIPS', 'Prscrbr_Type', 'Prscrbr_Type_Src', 'Brnd_Name', 'Gnrc_Name', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes']


In [4]:
import pandas as pd

path2 = r"C:\Users\Asus\OneDrive\Desktop\project_self_1\MUP_DPR_RY26_P04_V10_DY24_NPIBN.csv"

# Diabetes drug generic names (lowercase substrings to match)
diabetes_drugs = [
    'metformin', 'insulin', 'semaglutide', 'empagliflozin', 'dapagliflozin',
    'sitagliptin', 'dulaglutide', 'liraglutide', 'glipizide', 'glimepiride',
    'glyburide', 'pioglitazone', 'canagliflozin', 'linagliptin', 'tirzepatide',
    'glargine', 'aspart', 'lispro'   # common insulin variants
]
pattern = '|'.join(diabetes_drugs)

chunks = []
total_read = 0
for chunk in pd.read_csv(path2, chunksize=500_000, low_memory=False):
    total_read += len(chunk)
    mask = chunk['Gnrc_Name'].str.lower().str.contains(pattern, na=False)
    kept = chunk[mask]
    if len(kept) > 0:
        chunks.append(kept)
    print(f"Read {total_read:,} rows so far... kept {sum(len(c) for c in chunks):,}")

df_diabetes = pd.concat(chunks, ignore_index=True)
print("\nDONE. Diabetes-drug rows:", len(df_diabetes))

# Save the small filtered file so we never touch the 4GB monster again
df_diabetes.to_csv(r"C:\Users\Asus\OneDrive\Desktop\project_self_1\diabetes_prescribing.csv", index=False)
print("Saved as diabetes_prescribing.csv")

Read 500,000 rows so far... kept 40,256
Read 1,000,000 rows so far... kept 80,389
Read 1,500,000 rows so far... kept 120,485
Read 2,000,000 rows so far... kept 160,616
Read 2,500,000 rows so far... kept 199,943
Read 3,000,000 rows so far... kept 240,336
Read 3,500,000 rows so far... kept 280,068
Read 4,000,000 rows so far... kept 320,106
Read 4,500,000 rows so far... kept 360,578
Read 5,000,000 rows so far... kept 400,691
Read 5,500,000 rows so far... kept 440,826
Read 6,000,000 rows so far... kept 480,581
Read 6,500,000 rows so far... kept 520,905
Read 7,000,000 rows so far... kept 560,590
Read 7,500,000 rows so far... kept 599,934
Read 8,000,000 rows so far... kept 640,528
Read 8,500,000 rows so far... kept 680,806
Read 9,000,000 rows so far... kept 720,581
Read 9,500,000 rows so far... kept 760,659
Read 10,000,000 rows so far... kept 800,954
Read 10,500,000 rows so far... kept 840,966
Read 11,000,000 rows so far... kept 880,736
Read 11,500,000 rows so far... kept 920,289
Read 12,000

MemoryError: Unable to allocate 3.81 MiB for an array with shape (500000,) and data type float64

In [5]:
import pandas as pd

# Load the clean diabetes file
dia = pd.read_csv(r"C:\Users\Asus\OneDrive\Desktop\project_self_1\diabetes_prescribing.csv")

print("Total diabetes prescribing rows:", len(dia))
print("Unique prescribers:", dia['Prscrbr_NPI'].nunique())
print()

# Aggregate to ONE row per prescriber: total diabetes claims across all diabetes drugs
prescriber_dia = dia.groupby('Prscrbr_NPI').agg(
    total_diabetes_claims=('Tot_Clms', 'sum'),
    total_diabetes_cost=('Tot_Drug_Cst', 'sum'),
    n_diabetes_drugs=('Gnrc_Name', 'nunique'),
    specialty=('Prscrbr_Type', 'first'),
    state=('Prscrbr_State_Abrvtn', 'first')
).reset_index()

print("Prescribers after aggregation:", len(prescriber_dia))
print()
print("Diabetes claims distribution:")
print(prescriber_dia['total_diabetes_claims'].describe())

C:\Users\Asus\AppData\Local\Temp\ipykernel_26376\3237563483.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  dia = pd.read_csv(r"C:\Users\Asus\OneDrive\Desktop\project_self_1\diabetes_prescribing.csv")


Total diabetes prescribing rows: 2236487
Unique prescribers: 316969

Prescribers after aggregation: 316969

Diabetes claims distribution:
count    316969.000000
mean        328.799857
std         533.697591
min          11.000000
25%          43.000000
50%         150.000000
75%         413.000000
max       36213.000000
Name: total_diabetes_claims, dtype: float64


In [6]:
# Does SPECIALTY predict diabetes prescribing volume?
specialty_volume = prescriber_dia.groupby('specialty').agg(
    n_prescribers=('Prscrbr_NPI', 'count'),
    avg_diabetes_claims=('total_diabetes_claims', 'mean')
).sort_values('avg_diabetes_claims', ascending=False)

# Show specialties with at least 100 prescribers (reliable averages)
print("Avg diabetes claims by specialty (specialties with 100+ prescribers):")
print(specialty_volume[specialty_volume['n_prescribers'] >= 100].head(15).round(0))

Avg diabetes claims by specialty (specialties with 100+ prescribers):
                                                  n_prescribers  \
specialty                                                         
Endocrinology                                              6076   
General Practice                                           4719   
Internal Medicine                                         63012   
Family Practice                                           82336   
Geriatric Medicine                                         1402   
Osteopathic Manipulative Medicine                           218   
Pharmacist                                                 1371   
Certified Clinical Nurse Specialist                         296   
Emergency Medicine                                         1183   
Physician Assistant                                       26886   
Hospice and Palliative Care                                 177   
Nurse Practitioner                                        8

In [7]:
import pandas as pd
import sqlite3

folder = r"C:\Users\Asus\OneDrive\Desktop\project_self_1"

# 1. The diabetes volume per prescriber (we already built this, but reload clean from the saved file)
dia = pd.read_csv(folder + r"\diabetes_prescribing.csv")
prescriber_dia = dia.groupby('Prscrbr_NPI').agg(
    total_diabetes_claims=('Tot_Clms', 'sum'),
    total_diabetes_cost=('Tot_Drug_Cst', 'sum'),
    n_diabetes_drugs=('Gnrc_Name', 'nunique')
).reset_index()

# 2. The provider PROFILE file (characteristics) - load useful columns only
profile_cols = ['PRSCRBR_NPI','Prscrbr_Type','Prscrbr_State_Abrvtn','Prscrbr_RUCA_Desc',
                'Tot_Benes','Bene_Avg_Age','Bene_Avg_Risk_Scre','Tot_Clms','Tot_Drug_Cst']
profile = pd.read_csv(folder + r"\MUP_DPR_RY26_P04_V10_DY24_NPI.csv", usecols=profile_cols)
# rename the NPI column to match (one file uses PRSCRBR_NPI, casing matters)
profile = profile.rename(columns={'PRSCRBR_NPI':'Prscrbr_NPI',
                                   'Tot_Clms':'Tot_Clms_AllDrugs',
                                   'Tot_Drug_Cst':'Tot_Cost_AllDrugs'})

# 3. Load both into SQLite
conn = sqlite3.connect(":memory:")
prescriber_dia.to_sql("diabetes", conn, index=False, if_exists="replace")
profile.to_sql("profile", conn, index=False, if_exists="replace")

print("diabetes table:", len(prescriber_dia), "prescribers")
print("profile table:", len(profile), "prescribers")

C:\Users\Asus\AppData\Local\Temp\ipykernel_26376\2429092583.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  dia = pd.read_csv(folder + r"\diabetes_prescribing.csv")


diabetes table: 316969 prescribers
profile table: 1416883 prescribers


In [8]:
query = """
SELECT
    d.Prscrbr_NPI,
    p.Prscrbr_Type              AS specialty,
    p.Prscrbr_State_Abrvtn      AS state,
    p.Prscrbr_RUCA_Desc         AS rural_urban,
    p.Tot_Benes                 AS total_patients,
    p.Bene_Avg_Age              AS avg_patient_age,
    p.Bene_Avg_Risk_Scre        AS avg_patient_risk,
    p.Tot_Clms_AllDrugs         AS total_claims_all_drugs,
    d.total_diabetes_claims,
    d.total_diabetes_cost,
    d.n_diabetes_drugs
FROM diabetes d
INNER JOIN profile p ON d.Prscrbr_NPI = p.Prscrbr_NPI
"""

model_df = pd.read_sql(query, conn)

print("Joined table shape:", model_df.shape)
print("Matched prescribers:", len(model_df), "of", len(prescriber_dia))
print()
model_df.head()

Joined table shape: (316969, 11)
Matched prescribers: 316969 of 316969



,Prscrbr_NPI,specialty,state,rural_urban,total_patients,avg_patient_age,avg_patient_risk,total_claims_all_drugs,total_diabetes_claims,total_diabetes_cost,n_diabetes_drugs
0,1003000530,Internal Medicine,PA,Metropolitan area core: primary flow within an...,637.0,73.954474,1.272834,7892,412,105237.82,8
1,1003000902,Family Practice,KY,Metropolitan area core: primary flow within an...,534.0,71.717228,1.517527,7832,417,93876.17,8
2,1003000936,Cardiology,SC,Micropolitan high commuting: primary flow 30% ...,269.0,73.888476,1.538586,1370,14,10566.10,1
3,1003001256,Family Practice,CO,Metropolitan area core: primary flow within an...,90.0,69.777778,2.013294,572,35,12581.16,2
4,1003002049,Endocrinology,CA,Metropolitan area core: primary flow within an...,146.0,73.678082,1.643569,1589,837,611616.36,15


In [9]:
print("Missing values per column:")
print(model_df.isnull().sum())
print()
print("Rows with complete data:", model_df.dropna().shape[0], "of", len(model_df))

Missing values per column:
Prscrbr_NPI                  0
specialty                    1
state                        0
rural_urban                160
total_patients            1631
avg_patient_age              0
avg_patient_risk             0
total_claims_all_drugs       0
total_diabetes_claims        0
total_diabetes_cost          0
n_diabetes_drugs             0
dtype: int64

Rows with complete data: 315179 of 316969


In [10]:
# specialty: only 1 missing — drop that row (can't model without it)
# rural_urban: 160 missing — fill with "Unknown" (it's a category)
# total_patients: 1,631 missing — these are likely very small practices; drop them
#   (a doctor with suppressed patient count is a tiny outlier, safe to remove)

model_df = model_df.dropna(subset=["specialty", "total_patients"]).copy()
model_df["rural_urban"] = model_df["rural_urban"].fillna("Unknown")

print("After cleaning:", model_df.shape)
print("Remaining missing:", model_df.isnull().sum().sum())

After cleaning: (315337, 11)
Remaining missing: 0


In [11]:
import numpy as np

# TARGET: diabetes prescribing volume
# We predict the LOG of claims, because the distribution is very skewed
# (most doctors low, a few enormous). Log makes it learnable and is standard for counts.
model_df["log_diabetes_claims"] = np.log1p(model_df["total_diabetes_claims"])

# FEATURES: the doctor's profile (NOT diabetes volume — that's what we predict)
feature_cols = [
    "specialty", "state", "rural_urban",      # categorical
    "total_patients", "avg_patient_age",       # practice size & age
    "avg_patient_risk",                        # how sick their patients are
    "total_claims_all_drugs"                   # overall prescribing activity
]

X = model_df[feature_cols].copy()
y = model_df["log_diabetes_claims"].copy()

# One-hot encode the categoricals
X = pd.get_dummies(X, columns=["specialty", "state", "rural_urban"], drop_first=True)

print("Feature matrix shape:", X.shape)
print("Target (log claims) range:", round(y.min(),2), "to", round(y.max(),2))

Feature matrix shape: (315337, 196)
Target (log claims) range: 2.48 to 10.5


In [12]:
import re

# Clean feature column names: replace any [, ], <, >, and other problem chars
X.columns = [re.sub(r'[\[\]<>]', '_', str(c)) for c in X.columns]
# Also strip any other non-alphanumeric weirdness that can trip XGBoost
X.columns = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in X.columns]

# Check for any duplicate names created by the cleaning (can happen if two names collapse)
dupes = X.columns[X.columns.duplicated()].tolist()
print("Duplicate column names after cleaning:", dupes if dupes else "none")
print("Total columns:", len(X.columns))

Duplicate column names after cleaning: none
Total columns: 196


In [13]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R²:  {r2_score(y_test, y_pred):.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")

R²:  0.823
MAE: 0.421


In [14]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error

# CHECK 1: Is 0.82 stable, or a lucky split? (5-fold cross-validation)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print("CHECK 1 - Cross-validation R² across 5 folds:")
print("  scores:", cv_scores.round(3))
print(f"  mean {cv_scores.mean():.3f}, std {cv_scores.std():.3f}")
print()

# CHECK 2: Is the model just cheating off 'total_claims_all_drugs'?
# (a doctor who prescribes a lot overall obviously prescribes more diabetes drugs)
# Retrain WITHOUT it and see how much R² drops
X_no_total = X.drop(columns=[c for c in X.columns if c == 'total_claims_all_drugs'])
Xtr2, Xte2, ytr2, yte2 = train_test_split(X_no_total, y, test_size=0.2, random_state=42)
from xgboost import XGBRegressor
m2 = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, random_state=42)
m2.fit(Xtr2, ytr2)
print("CHECK 2 - R² WITHOUT 'total_claims_all_drugs':", round(r2_score(yte2, m2.predict(Xte2)), 3))
print("  (if this crashes to ~0.3, the model was mostly riding overall volume)")
print()

# CHECK 3: What features actually drive it?
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("CHECK 3 - Top 8 features:")
print(imp.head(8).round(3).to_string())

CHECK 1 - Cross-validation R² across 5 folds:
  scores: [0.82  0.821 0.821 0.817 0.822]
  mean 0.820, std 0.002

CHECK 2 - R² WITHOUT 'total_claims_all_drugs': 0.608
  (if this crashes to ~0.3, the model was mostly riding overall volume)

CHECK 3 - Top 8 features:
total_claims_all_drugs                                                      0.181
specialty_Endocrinology                                                     0.138
specialty_Cardiology                                                        0.117
specialty_Interventional_Cardiology                                         0.055
specialty_Psychiatry                                                        0.036
specialty_Internal_Medicine                                                 0.035
specialty_Rheumatology                                                      0.029
specialty_Student_in_an_Organized_Health_Care_Education_Training_Program    0.028


In [15]:
import pandas as pd

folder = r"C:\Users\Asus\OneDrive\Desktop\project_self_1"

# Reload profile with MANY more real features (patient demographics, beneficiary mix)
rich_cols = [
    'PRSCRBR_NPI','Prscrbr_Type','Prscrbr_State_Abrvtn','Prscrbr_RUCA_Desc',
    'Tot_Benes','Tot_Clms','Tot_Drug_Cst','Tot_30day_Fills','Tot_Day_Suply',
    'Bene_Avg_Age','Bene_Avg_Risk_Scre',
    # patient age breakdown
    'Bene_Age_LT_65_Cnt','Bene_Age_65_74_Cnt','Bene_Age_75_84_Cnt','Bene_Age_GT_84_Cnt',
    # patient sex + dual-eligible (poverty proxy, matters for drug access)
    'Bene_Feml_Cnt','Bene_Male_Cnt','Bene_Dual_Cnt','Bene_Ndual_Cnt',
    # overall prescribing behavior signals
    'Brnd_Tot_Clms','Gnrc_Tot_Clms','GE65_Tot_Clms'
]
profile_rich = pd.read_csv(folder + r"\MUP_DPR_RY26_P04_V10_DY24_NPI.csv", usecols=rich_cols)
profile_rich = profile_rich.rename(columns={'PRSCRBR_NPI':'Prscrbr_NPI'})

print("Rich profile columns loaded:", profile_rich.shape[1])
print("Rows:", len(profile_rich))
print("\nMissing values in new features:")
print(profile_rich.isnull().sum()[profile_rich.isnull().sum() > 0])

Rich profile columns loaded: 22
Rows: 1416883

Missing values in new features:
Prscrbr_RUCA_Desc       1234
Prscrbr_Type               2
Tot_Benes             145511
GE65_Tot_Clms         335561
Brnd_Tot_Clms         622278
Gnrc_Tot_Clms          31303
Bene_Age_LT_65_Cnt    799191
Bene_Age_65_74_Cnt    718012
Bene_Age_75_84_Cnt    857610
Bene_Age_GT_84_Cnt    965581
Bene_Feml_Cnt         373430
Bene_Male_Cnt         373430
Bene_Dual_Cnt         558836
Bene_Ndual_Cnt        558836
Bene_Avg_Risk_Scre         1
dtype: int64


In [16]:
import numpy as np

# Predict expected volume for ALL doctors (using the model on the full feature set)
model_df["predicted_log"] = model.predict(X)
model_df["predicted_claims"] = np.expm1(model_df["predicted_log"])
model_df["actual_claims"] = model_df["total_diabetes_claims"]

# The GAP: actual minus expected
# Negative gap = prescribing LESS than their profile predicts = OPPORTUNITY
model_df["gap"] = model_df["actual_claims"] - model_df["predicted_claims"]

# Opportunity = how many MORE claims they'd write if they matched their expected level
model_df["opportunity"] = -model_df["gap"]   # positive = under-prescribing

# Look at the biggest opportunities (under-prescribers)
targets = model_df.sort_values("opportunity", ascending=False)
print("Top 10 targeting opportunities (under-prescribing doctors):")
print(targets[["specialty","state","total_patients","avg_patient_risk",
               "actual_claims","predicted_claims","opportunity"]].head(10).round(0).to_string(index=False))

Top 10 targeting opportunities (under-prescribing doctors):
        specialty state  total_patients  avg_patient_risk  actual_claims  predicted_claims  opportunity
    Endocrinology    IL           970.0               1.0           1277            4950.0       3673.0
    Endocrinology    OK          1403.0               2.0           3133            5768.0       2635.0
  Family Practice    FL         30505.0               1.0             28            2595.0       2567.0
    Endocrinology    FL           626.0               2.0           1609            4162.0       2553.0
  Family Practice    NJ         26721.0               1.0             30            2273.0       2243.0
    Endocrinology    PR           825.0               1.0           2846            5058.0       2212.0
    Endocrinology    PR          1907.0               2.0           7211            9405.0       2194.0
    Endocrinology    PR          1255.0               2.0           3694            5818.0       2124.0
Inte

In [17]:
import numpy as np

# Filter out likely data artifacts before ranking opportunities:
# 1. Absurd patient counts (organizational NPIs, not individual prescribers)
# 2. Doctors with tiny actual prescribing but huge predicted (often data quirks)

# Reasonable individual-prescriber bounds
clean_targets = model_df[
    (model_df["total_patients"] <= 5000) &          # individual docs, not org NPIs
    (model_df["actual_claims"] >= 50)                # must have SOME real prescribing history
].copy()

# Also: express opportunity as a RATIO too, not just absolute
# (prescribing 40% of expected is more actionable signal than raw gap)
clean_targets["pct_of_expected"] = (
    clean_targets["actual_claims"] / clean_targets["predicted_claims"] * 100
)

# Re-rank: biggest absolute opportunity among CREDIBLE targets
targets = clean_targets.sort_values("opportunity", ascending=False)
print("Top 10 CREDIBLE targeting opportunities:")
print(targets[["specialty","state","total_patients","avg_patient_risk",
               "actual_claims","predicted_claims","opportunity","pct_of_expected"]]
      .head(10).round(0).to_string(index=False))

Top 10 CREDIBLE targeting opportunities:
        specialty state  total_patients  avg_patient_risk  actual_claims  predicted_claims  opportunity  pct_of_expected
    Endocrinology    IL           970.0               1.0           1277            4950.0       3673.0             26.0
    Endocrinology    OK          1403.0               2.0           3133            5768.0       2635.0             54.0
    Endocrinology    FL           626.0               2.0           1609            4162.0       2553.0             39.0
    Endocrinology    PR           825.0               1.0           2846            5058.0       2212.0             56.0
    Endocrinology    PR          1907.0               2.0           7211            9405.0       2194.0             77.0
    Endocrinology    PR          1255.0               2.0           3694            5818.0       2124.0             63.0
Internal Medicine    PR          1799.0               2.0            598            2605.0       2007.0         

In [18]:
# THE DECISION: given limited rep capacity, which doctors to target?
# Rank by opportunity, take the top N, show the total prescription upside captured.

total_opportunity = clean_targets["opportunity"].clip(lower=0).sum()

print(f"Total addressable opportunity (all under-prescribers): {total_opportunity:,.0f} claims\n")
print(f"{'Reps can visit':>15} | {'Opportunity captured':>20} | {'% of total':>10}")
print("-"*52)
for n in [100, 250, 500, 1000, 2500]:
    captured = clean_targets.nlargest(n, "opportunity")["opportunity"].clip(lower=0).sum()
    pct = captured / total_opportunity * 100
    print(f"{n:>15} | {captured:>18,.0f}  | {pct:>8.1f}%")

Total addressable opportunity (all under-prescribers): 7,730,357 claims

 Reps can visit | Opportunity captured | % of total
----------------------------------------------------
            100 |            133,072  |      1.7%
            250 |            259,066  |      3.4%
            500 |            426,783  |      5.5%
           1000 |            697,046  |      9.0%
           2500 |          1,301,442  |     16.8%


In [19]:
import numpy as np

# Work on the credible targets (already filtered for real individual prescribers)
t = clean_targets.copy()

# Only under-prescribers are targetable (positive opportunity)
t = t[t["opportunity"] > 0].copy()

# --- Build a smarter targeting score ---
# Raw opportunity alone favors huge practices. We weight it by PERSUADABILITY:
# a doctor at 26% of expected has more room/likelihood to move than one at 90%.
# persuadability = how far BELOW expected they are (0 to 1)
t["persuadability"] = 1 - (t["actual_claims"] / t["predicted_claims"])
t["persuadability"] = t["persuadability"].clip(0, 1)

# Targeting score = opportunity size × persuadability
# (big gap AND very persuadable = best target)
t["target_score"] = t["opportunity"] * t["persuadability"]

# Compare the two rankings to SEE the difference
print("Top 5 by RAW opportunity (old way):")
print(t.nlargest(5,"opportunity")[["specialty","state","actual_claims","predicted_claims","opportunity","persuadability","target_score"]].round(2).to_string(index=False))
print()
print("Top 5 by TARGET SCORE (new, ROI-weighted):")
print(t.nlargest(5,"target_score")[["specialty","state","actual_claims","predicted_claims","opportunity","persuadability","target_score"]].round(2).to_string(index=False))

Top 5 by RAW opportunity (old way):
    specialty state  actual_claims  predicted_claims  opportunity  persuadability  target_score
Endocrinology    IL           1277       4950.180176      3673.18            0.74       2725.61
Endocrinology    OK           3133       5768.290039      2635.29            0.46       1203.96
Endocrinology    FL           1609       4162.279785      2553.28            0.61       1566.27
Endocrinology    PR           2846       5057.990234      2211.99            0.44        967.36
Endocrinology    PR           7211       9404.719727      2193.72            0.23        511.70

Top 5 by TARGET SCORE (new, ROI-weighted):
        specialty state  actual_claims  predicted_claims  opportunity  persuadability  target_score
    Endocrinology    IL           1277       4950.180176      3673.18            0.74       2725.61
Internal Medicine    NY            136       1834.010010      1698.01            0.93       1572.10
    Endocrinology    FL           1609      

In [20]:
# Cut the target_score into actionable tiers for the sales team
# Use quantiles so tiers are relative to the population of under-prescribers
t["tier"] = pd.qcut(
    t["target_score"],
    q=[0, 0.5, 0.8, 0.95, 1.0],
    labels=["Low", "Medium", "High", "Priority"]
)

print("Doctors per tier:")
print(t["tier"].value_counts().sort_index())
print()
print("Avg opportunity & persuadability by tier:")
print(t.groupby("tier", observed=True).agg(
    n_doctors=("Prscrbr_NPI","count"),
    avg_opportunity=("opportunity","mean"),
    avg_persuadability=("persuadability","mean"),
    total_opportunity=("opportunity","sum")
).round(2))

Doctors per tier:
tier
Low         44085
Medium      26451
High        13225
Priority     4409
Name: count, dtype: int64

Avg opportunity & persuadability by tier:
          n_doctors  avg_opportunity  avg_persuadability  total_opportunity
tier                                                                       
Low           44085            27.34                0.11         1205198.59
Medium        26451            91.12                0.28         2410229.25
High          13225           180.97                0.40         2393284.59
Priority       4409           390.48                0.52         1721644.10


In [21]:
import joblib

folder = r"C:\Users\Asus\OneDrive\Desktop\project_self_1"

# 1. Save the trained model
joblib.dump(model, folder + r"\diabetes_targeting_model.pkl")
print("✓ Model saved: diabetes_targeting_model.pkl")

# 2. Save the feature column order (CRITICAL - the model needs columns in the exact same order)
joblib.dump(list(X.columns), folder + r"\model_features.pkl")
print("✓ Feature list saved: model_features.pkl")

# 3. Save the full scored + tiered prescriber table (for the dashboard & app)
#    Include the identifying/profile columns + our computed scores + tiers
output_cols = [
    "Prscrbr_NPI", "specialty", "state", "rural_urban",
    "total_patients", "avg_patient_age", "avg_patient_risk",
    "actual_claims", "predicted_claims", "opportunity",
    "persuadability", "target_score", "tier"
]
t[output_cols].to_csv(folder + r"\prescriber_targets_scored.csv", index=False)
print("✓ Scored targets saved: prescriber_targets_scored.csv")
print(f"  ({len(t):,} scored under-prescribing doctors)")

✓ Model saved: diabetes_targeting_model.pkl
✓ Feature list saved: model_features.pkl
✓ Scored targets saved: prescriber_targets_scored.csv
  (88,170 scored under-prescribing doctors)


In [22]:
import json

# Export the exact category values the model knows, for the app's dropdowns
categories = {
    "specialty": sorted(model_df["specialty"].dropna().unique().tolist()),
    "state": sorted(model_df["state"].dropna().unique().tolist()),
    "rural_urban": sorted(model_df["rural_urban"].dropna().unique().tolist()),
}
# Also export sensible numeric defaults (medians) so the form pre-fills realistically
numeric_defaults = {
    "total_patients": float(model_df["total_patients"].median()),
    "avg_patient_age": float(model_df["avg_patient_age"].median()),
    "avg_patient_risk": float(model_df["avg_patient_risk"].median()),
    "total_claims_all_drugs": float(model_df["total_claims_all_drugs"].median()),
}
out = {"categories": categories, "numeric_defaults": numeric_defaults}

folder = r"C:\Users\Asus\OneDrive\Desktop\project_self_1"
with open(folder + r"\model_categories.json", "w") as f:
    json.dump(out, f, indent=2)

print("Saved model_categories.json")
print("Specialties:", len(categories["specialty"]))
print("States:", len(categories["state"]))
print("Rural/urban values:", len(categories["rural_urban"]))
print("Sample rural_urban:", categories["rural_urban"][:2])

Saved model_categories.json
Specialties: 121
States: 59
Rural/urban values: 15
Sample rural_urban: ['Metropolitan area core: primary flow within an urbanized area of 50,000 and greater', 'Metropolitan area high commuting: primary flow 30% or more to a urbanized area of 50,000 and greater']


In [23]:
import pandas as pd, json

folder = r"C:\Users\Asus\OneDrive\Desktop\project_self_1"
scored = pd.read_csv(folder + r"\prescriber_targets_scored.csv")

categories = {
    "specialty": sorted(scored["specialty"].dropna().unique().tolist()),
    "state": sorted(scored["state"].dropna().unique().tolist()),
    "rural_urban": sorted(scored["rural_urban"].dropna().unique().tolist()),
}
numeric_defaults = {
    "total_patients": float(scored["total_patients"].median()),
    "avg_patient_age": float(scored["avg_patient_age"].median()),
    "avg_patient_risk": float(scored["avg_patient_risk"].median()),
    # total_claims_all_drugs isn't in the scored file, so use a reasonable default
    "total_claims_all_drugs": 5000.0,
}
out = {"categories": categories, "numeric_defaults": numeric_defaults}

with open(folder + r"\model_categories.json", "w") as f:
    json.dump(out, f, indent=2)

print("Saved model_categories.json")
print("Specialties:", len(categories["specialty"]))
print("States:", len(categories["state"]))
print("Rural/urban values:", len(categories["rural_urban"]))
print("Sample rural_urban:", categories["rural_urban"][:2])

Saved model_categories.json
Specialties: 75
States: 58
Rural/urban values: 15
Sample rural_urban: ['Metropolitan area core: primary flow within an urbanized area of 50,000 and greater', 'Metropolitan area high commuting: primary flow 30% or more to a urbanized area of 50,000 and greater']


In [24]:
import pandas as pd, json

folder = r"C:\Users\Asus\OneDrive\Desktop\project_self_1"
scored = pd.read_csv(folder + r"\prescriber_targets_scored.csv")

# Take the top 10,000 targets by score (across all tiers), capped for browser performance
keep = scored.sort_values("target_score", ascending=False).head(10000).copy()

# Rename columns to what the app expects (match the sample data schema)
out = keep.rename(columns={"Prscrbr_NPI": "npi"})[[
    "npi","specialty","state","total_patients","avg_patient_risk",
    "actual_claims","predicted_claims","opportunity","persuadability","target_score","tier"
]]

# Round for cleanliness
for c in ["total_patients","actual_claims","predicted_claims","opportunity","target_score"]:
    out[c] = out[c].round(0).astype(int)
out["persuadability"] = out["persuadability"].round(2)
out["avg_patient_risk"] = out["avg_patient_risk"].round(2)

records = out.to_dict(orient="records")

# Save into the app's public folder so it can fetch it
app_public = folder + r"\prescriber-app\public\prescribers.json"
with open(app_public, "w") as f:
    json.dump(records, f)

print(f"Saved {len(records)} doctors to prescriber-app/public/prescribers.json")
print("Tiers:", out["tier"].value_counts().to_dict())
print("Sample record:", records[0])

Saved 10000 doctors to prescriber-app/public/prescribers.json
Tiers: {'High': 5591, 'Priority': 4409}
Sample record: {'npi': 1972512374, 'specialty': 'Endocrinology', 'state': 'IL', 'total_patients': 970, 'avg_patient_risk': 1.22, 'actual_claims': 1277, 'predicted_claims': 4950, 'opportunity': 3673, 'persuadability': 0.74, 'target_score': 2726, 'tier': 'Priority'}
